# Soluciones — Clase 27: Detección de Anomalías

> Este notebook contiene las soluciones completas a todos los ejercicios.
> Intenta resolverlos por tu cuenta antes de consultar aquí.

## Solución Ejercicio 1 — Identificación intuitiva

**1.** En la lista [22, 23, 21, 24, 22, **75**, 23, 22], el valor **75** es claramente una anomalía. Todos los demás valores están entre 21 y 24, y 75 está a más de 50 grados de diferencia. Podría ser un error de medición o una falla del termómetro.

**2.** Una venta de $50 cuando el promedio es $500-$2000 SÍ es una anomalía. Podría ser: una devolución registrada incorrectamente, un producto en oferta extrema, una transacción de prueba en el sistema, o un error de carga. Se debe investigar antes de actuar.

**3.** Un tiempo de espera de 30 segundos en urgencias es estadísticamente inusual (outlier) pero no es un problema — al contrario, es ideal. Una anomalía problemática sería un tiempo de 8 horas. La distinción entre outlier y anomalía siempre depende del **contexto y del impacto**.

In [ ]:
# Imports para todas las soluciones
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from scipy import stats
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

In [ ]:
# Solución Ejercicio 2 — Exploración del dataset de ventas
df_ventas = pd.read_csv('../../datasets/ventas_tienda.csv')

print('Forma:', df_ventas.shape)
print('\nEstadísticas descriptivas:')
print(df_ventas['total_neto'].describe())

# Diferencia entre max y percentil 95
p95 = df_ventas['total_neto'].quantile(0.95)
maximo = df_ventas['total_neto'].max()
print(f'\nPercentil 95: {p95:.2f}')
print(f'Máximo: {maximo:.2f}')
print(f'Diferencia: {maximo - p95:.2f}')
print('Si la diferencia es grande, hay outliers extremos en el top 5%.')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df_ventas['total_neto'].plot(ax=axes[0], color='steelblue', alpha=0.7)
axes[0].set_title('Ventas diarias')
df_ventas['total_neto'].plot(kind='box', ax=axes[1])
axes[1].set_title('Distribución (boxplot)')
plt.tight_layout()
plt.show()

In [ ]:
# Solución Ejercicio 3 — Método IQR
Q1 = df_ventas['total_neto'].quantile(0.25)
Q3 = df_ventas['total_neto'].quantile(0.75)
IQR = Q3 - Q1

limite_inf = Q1 - 1.5 * IQR
limite_sup = Q3 + 1.5 * IQR

print(f'Q1={Q1:.2f}, Q3={Q3:.2f}, IQR={IQR:.2f}')
print(f'Límites: [{limite_inf:.2f}, {limite_sup:.2f}]')

df_ventas['outlier_iqr'] = (
    (df_ventas['total_neto'] < limite_inf) |
    (df_ventas['total_neto'] > limite_sup)
)

n_iqr = df_ventas['outlier_iqr'].sum()
print(f'\nAnomalías IQR (1.5x): {n_iqr}')

# Con umbral más estricto (3x IQR)
limite_sup_3 = Q3 + 3 * IQR
limite_inf_3 = Q1 - 3 * IQR
n_iqr_3 = ((df_ventas['total_neto'] < limite_inf_3) | (df_ventas['total_neto'] > limite_sup_3)).sum()
print(f'Anomalías IQR (3.0x, más estricto): {n_iqr_3}')
print('Con 3x IQR solo detectamos los casos más extremos.')

In [ ]:
# Solución Ejercicio 4 — Z-score
z_scores = np.abs(stats.zscore(df_ventas['total_neto']))
df_ventas['z_score'] = z_scores
df_ventas['outlier_zscore'] = z_scores > 3

n_zscore = df_ventas['outlier_zscore'].sum()
print(f'Anomalías Z-score (|z|>3): {n_zscore}')

print('\nTop 5 por Z-score:')
print(df_ventas.nlargest(5, 'z_score')[['total_neto', 'z_score']].to_string())

ambos = (df_ventas['outlier_iqr'] & df_ventas['outlier_zscore']).sum()
solo_iqr = (df_ventas['outlier_iqr'] & ~df_ventas['outlier_zscore']).sum()
solo_zscore = (~df_ventas['outlier_iqr'] & df_ventas['outlier_zscore']).sum()
print(f'\nDetectados por ambos: {ambos}')
print(f'Solo IQR: {solo_iqr}')
print(f'Solo Z-score: {solo_zscore}')
print('\nEl Z-score es más sensible a la distribución de los datos.')
print('Si hay outliers extremos, afectan la media y desviación, haciendo el Z-score menos robusto.')
print('El IQR es más robusto con distribuciones asimétricas.')

In [ ]:
# Solución Ejercicio 5 — Isolation Forest
iso_forest = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
df_ventas['anomalia_iso'] = iso_forest.fit_predict(df_ventas[['total_neto']])

n_iso = (df_ventas['anomalia_iso'] == -1).sum()
print(f'Isolation Forest (contamination=0.05): {n_iso} anomalías')

# Con contamination más alta
iso2 = IsolationForest(n_estimators=100, contamination=0.10, random_state=42)
pred2 = iso2.fit_predict(df_ventas[['total_neto']])
n_iso2 = (pred2 == -1).sum()
print(f'Isolation Forest (contamination=0.10): {n_iso2} anomalías')
print(f'Diferencia: {n_iso2 - n_iso} anomalías adicionales al subir contamination')

# Visualización
fig, ax = plt.subplots(figsize=(14, 5))
normales = df_ventas[df_ventas['anomalia_iso'] == 1]
anomalas = df_ventas[df_ventas['anomalia_iso'] == -1]
ax.scatter(normales.index, normales['total_neto'], c='steelblue', s=15, alpha=0.5, label='Normal')
ax.scatter(anomalas.index, anomalas['total_neto'], c='red', s=60, zorder=5, label=f'Anomalía ({n_iso})')
ax.axhline(limite_sup, color='orange', linestyle='--', alpha=0.7, label='Límite IQR')
ax.set_title('Isolation Forest — Anomalías en ventas')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Solución Ejercicio 6 — LOF en transporte
df_transporte = pd.read_csv('../../datasets/transporte.csv')
print('Retrasos de transporte:')
print(df_transporte['retraso_min'].describe())

lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05)
df_transporte['anomalia_lof'] = lof.fit_predict(df_transporte[['retraso_min']])
df_transporte['lof_score'] = lof.negative_outlier_factor_

n_lof = (df_transporte['anomalia_lof'] == -1).sum()
print(f'\nAnomalías LOF: {n_lof}')

print('\nTop 5 rutas más anómalas:')
print(df_transporte.nsmallest(5, 'lof_score')[['retraso_min', 'lof_score']].to_string())

print('\nNota: Los retrasos anómalos suelen ser los más ALTOS.')
print('Pero LOF también puede detectar retrasos inusualmente BAJOS si son extraños en contexto local.')
print('Explicaciones posibles para retrasos extremos: accidente, condiciones climáticas, falla mecánica.')

In [ ]:
# Solución Ejercicio 7 — Comparación de todos los métodos
comparacion = pd.DataFrame({
    'total_neto': df_ventas['total_neto'],
    'IQR': df_ventas['outlier_iqr'].astype(int),
    'Z-score': df_ventas['outlier_zscore'].astype(int),
    'Isolation Forest': (df_ventas['anomalia_iso'] == -1).astype(int)
})
comparacion['votos'] = comparacion[['IQR', 'Z-score', 'Isolation Forest']].sum(axis=1)

print('Detecciones por método:')
print(comparacion[['IQR', 'Z-score', 'Isolation Forest']].sum().to_string())

print('\nPuntos con consenso (3/3 métodos):')
print(comparacion[comparacion['votos'] == 3][['total_neto', 'IQR', 'Z-score', 'Isolation Forest']].to_string())

# Visualización comparativa
metodos = ['IQR', 'Z-score', 'Isolation Forest']
counts = [comparacion['IQR'].sum(), comparacion['Z-score'].sum(),
          comparacion['Isolation Forest'].sum()]

plt.figure(figsize=(7, 4))
bars = plt.bar(metodos, counts, color=['#3498db', '#e74c3c', '#2ecc71'], edgecolor='white')
for bar, count in zip(bars, counts):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             str(count), ha='center', fontweight='bold')
plt.title('Anomalías detectadas por método')
plt.ylabel('Cantidad')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('\nCriterio recomendado: reportar los puntos con al menos 2/3 votos.')
print('Siempre validar con expertos del dominio antes de tomar acciones.')

In [ ]:
# Solución Ejercicio 8 — Detección multivariada
columnas_numericas = df_ventas.select_dtypes(include='number').columns.tolist()
# Excluir columnas de resultado de anomalías ya generadas
cols_para_modelo = [c for c in columnas_numericas
                    if c not in ['outlier_iqr', 'z_score', 'outlier_zscore', 'anomalia_iso']]

print('Columnas numéricas disponibles para modelo multivariado:')
print(cols_para_modelo)

if len(cols_para_modelo) >= 2:
    iso_multi = IsolationForest(contamination=0.05, random_state=42)
    df_ventas['anomalia_multi'] = iso_multi.fit_predict(df_ventas[cols_para_modelo])
    
    n_multi = (df_ventas['anomalia_multi'] == -1).sum()
    print(f'\nAnomalías multivariadas: {n_multi}')
    
    diff = (df_ventas['anomalia_iso'] == -1) != (df_ventas['anomalia_multi'] == -1)
    print(f'Diferencias vs univariado: {diff.sum()} puntos')
    print('\nLa detección multivariada puede encontrar anomalías que ninguna variable')
    print('revela por sí sola: ej. ventas altas + cantidad baja = precio inusualmente alto.')
else:
    print('El dataset tiene solo una variable numérica disponible para detección multivariada.')